# 📘 Week 2 · Day 12-14: Feature Engineering — Kaggle의 핵심

## 🎯 학습 목표
- Kaggle에서 **등수를 바꾸는** 피처 엔지니어링 패턴을 모두 익힌다
- **결측치 처리, 인코딩, 스케일링, 상호작용, Target Encoding** 마스터
- **시계열·텍스트·카운트** 피처의 실전 트릭
- **Data Leakage**를 유발하지 않는 **올바른 적용 순서**

> 🔥 "More data beats clever algorithms, but better features beat more data." — Kaggle 격언


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')
np.random.seed(0)

---

## 1. 결측치 처리 (Missing Values)

### 전략 종류

| 전략 | 언제 | 코드 |
|------|------|------|
| **제거** | 결측 < 5% | `df.dropna()` |
| **상수** | 결측이 의미있음 | `.fillna(0)` or `.fillna('missing')` |
| **통계** | 일반적 | median, mean, mode |
| **그룹별** | 더 정확 | `groupby(...).transform(...)` |
| **모델 예측** | 정교함 필요 | IterativeImputer, KNN Imputer |
| **결측 플래그** | 결측 자체가 신호 | `isna().astype(int)` 새 컬럼 |


In [ ]:
df = pd.DataFrame({
    'age':    [25, np.nan, 35, np.nan, 45, 55, np.nan, 28],
    'fare':   [7.2, 71.2, 8.1, 53.1, 8.1, np.nan, 30.5, 10.5],
    'pclass': [3, 1, 3, 1, 3, 2, 2, 3],
    'sex':    ['M', 'F', 'F', 'M', 'M', 'F', 'F', 'M']
})
df

In [ ]:
# 전략 1: 단순 fillna
df1 = df.copy()
df1['age']  = df1['age'].fillna(df1['age'].median())
df1['fare'] = df1['fare'].fillna(df1['fare'].mean())
df1

In [ ]:
# 전략 2: 그룹별 결측치 채우기 (훨씬 정확)
df2 = df.copy()
df2['age'] = df2.groupby(['pclass', 'sex'])['age'].transform(lambda x: x.fillna(x.median()))
df2

In [ ]:
# 전략 3: 결측 플래그 추가 (결측 자체가 정보)
df3 = df.copy()
df3['age_missing'] = df3['age'].isna().astype(int)
df3['age'] = df3['age'].fillna(df3['age'].median())
df3

In [ ]:
# 전략 4: 고급 — IterativeImputer (다른 피처로 예측하여 채움)
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

num_cols = ['age', 'fare', 'pclass']
imputer = IterativeImputer(random_state=0)
df4 = df.copy()
df4[num_cols] = imputer.fit_transform(df4[num_cols])
df4

---

## 2. 범주형 인코딩 (Categorical Encoding)

### 2.1 Label Encoding
- 순서 있는 범주 (low/mid/high)
- 트리 모델용 (XGBoost 등은 숫자만 받음)

In [ ]:
from sklearn.preprocessing import LabelEncoder

df_cat = pd.DataFrame({'grade': ['A', 'B', 'C', 'A', 'B', 'D', 'C']})
le = LabelEncoder()
df_cat['grade_le'] = le.fit_transform(df_cat['grade'])
df_cat

### 2.2 One-Hot Encoding
- 순서 없는 범주 (도시, 색상)
- 선형모델/신경망에 필수

In [ ]:
df_city = pd.DataFrame({'city': ['NY', 'LA', 'SF', 'NY', 'LA']})
pd.get_dummies(df_city, drop_first=True)

### 2.3 Frequency Encoding
- 고카디널리티 범주 (수천 개 이상) 처리에 좋음
- 빈도 자체가 정보가 될 때

In [ ]:
df_fe = pd.DataFrame({'zipcode': ['10001', '10001', '90001', '90001', '90001', '30301']})
freq_map = df_fe['zipcode'].value_counts().to_dict()
df_fe['zip_freq'] = df_fe['zipcode'].map(freq_map)
df_fe

### 2.4 Target Encoding (가장 강력하지만 위험)

**아이디어**: 범주의 평균 타깃값을 사용

⚠️ **Data Leakage 주의** → **KFold로 out-of-fold 방식**이 필수

In [ ]:
# 나쁜 예시 (Leakage 발생!)
df_bad = pd.DataFrame({
    'city':   ['NY', 'NY', 'LA', 'LA', 'SF', 'SF', 'NY'],
    'target': [1,    1,    0,    1,    0,    0,    1]
})

# 💀 전체 평균 사용 → test 데이터의 타깃을 학습에 반영
leak_map = df_bad.groupby('city')['target'].mean()
df_bad['city_enc_BAD'] = df_bad['city'].map(leak_map)
print("Leakage 있는 방식:")
print(df_bad)

In [ ]:
# ✅ 올바른 방식: K-Fold out-of-fold target encoding
def kfold_target_encoding(train, test, col, target, n_splits=5, smoothing=10):
    train = train.copy(); test = test.copy()
    global_mean = train[target].mean()

    train_enc = np.zeros(len(train))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=0)

    for tr_idx, val_idx in kf.split(train):
        tr_fold = train.iloc[tr_idx]
        # fold 내부에서만 통계 계산
        agg = tr_fold.groupby(col)[target].agg(['mean', 'count']).reset_index()
        # smoothing
        agg['smoothed'] = (agg['count']*agg['mean'] + smoothing*global_mean) / (agg['count'] + smoothing)
        mapping = dict(zip(agg[col], agg['smoothed']))
        train_enc[val_idx] = train.iloc[val_idx][col].map(mapping).fillna(global_mean)

    # test는 전체 train 통계 사용
    agg_all = train.groupby(col)[target].agg(['mean', 'count']).reset_index()
    agg_all['smoothed'] = (agg_all['count']*agg_all['mean'] + smoothing*global_mean) / (agg_all['count'] + smoothing)
    mapping_all = dict(zip(agg_all[col], agg_all['smoothed']))
    test_enc = test[col].map(mapping_all).fillna(global_mean).values

    return train_enc, test_enc

# 사용 예시
np.random.seed(0)
n = 1000
train = pd.DataFrame({
    'city':   np.random.choice(['NY', 'LA', 'SF', 'CHI', 'BOS'], n),
    'target': np.random.binomial(1, 0.3, n)
})
test = pd.DataFrame({'city': np.random.choice(['NY', 'LA', 'SF', 'CHI', 'BOS'], 200)})

train['city_te'], test['city_te'] = kfold_target_encoding(train, test, 'city', 'target')
print(train.head())
print("\nTest encoded sample:", test['city_te'][:5].values)

---

## 3. 스케일링 (Scaling)

| 방법 | 공식 | 언제 |
|------|------|------|
| **StandardScaler** | `(x - μ) / σ` | 대부분 (정규분포 가정) |
| **MinMaxScaler** | `(x - min) / (max - min)` | 0~1 범위 필요, 이미지 |
| **RobustScaler** | `(x - median) / IQR` | **outlier 많을 때** |
| **PowerTransformer** | Box-Cox / Yeo-Johnson | 왜도 심할 때 정규화 |

> ⚠️ 트리 기반(RF, XGBoost, LGBM)은 스케일링 **불필요**

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, PowerTransformer

data = np.random.exponential(2, 1000).reshape(-1, 1)

fig, axes = plt.subplots(1, 5, figsize=(18, 3))
axes[0].hist(data, bins=40, color='gray');  axes[0].set_title('Original')
for ax, (name, sc) in zip(axes[1:], [
    ('Standard',  StandardScaler()),
    ('MinMax',    MinMaxScaler()),
    ('Robust',    RobustScaler()),
    ('PowerYeo',  PowerTransformer(method='yeo-johnson'))
]):
    ax.hist(sc.fit_transform(data), bins=40, color='steelblue')
    ax.set_title(name)
plt.tight_layout(); plt.show()

---

## 4. 수치형 변환

### 4.1 Log / Box-Cox - 왜도 줄이기

In [ ]:
fare = np.random.exponential(30, 1000)
fig, ax = plt.subplots(1, 3, figsize=(14, 3))
ax[0].hist(fare, bins=40); ax[0].set_title('original (skewed)')
ax[1].hist(np.log1p(fare), bins=40, color='green'); ax[1].set_title('log1p')
ax[2].hist(np.sqrt(fare), bins=40, color='orange'); ax[2].set_title('sqrt')
plt.tight_layout(); plt.show()

### 4.2 Binning - 구간화

In [ ]:
ages = np.random.randint(0, 80, 1000)

# 동일 너비
bins_width = pd.cut(ages, bins=5, labels=['kid','young','adult','mid','senior'])

# 동일 빈도 (quantile)
bins_qcut = pd.qcut(ages, q=5, labels=['q1','q2','q3','q4','q5'])

print("Width-based:"); print(pd.Series(bins_width).value_counts())
print("\nQuantile-based:"); print(pd.Series(bins_qcut).value_counts())

---

## 5. 상호작용 피처 (Interaction Features) — Kaggle 단골

In [ ]:
df = pd.DataFrame({
    'age':    [25, 32, 45, 28, 55, 60],
    'income': [30, 50, 80, 35, 90, 100],
    'sex':    ['M', 'F', 'M', 'F', 'M', 'F']
})

# 수치 × 수치 (합, 곱, 비율, 차이)
df['age_income_ratio'] = df['age'] / df['income']
df['age_plus_income']  = df['age'] + df['income']
df['age_x_income']     = df['age'] * df['income']
df['age_minus_income'] = df['age'] - df['income']

# 수치 × 범주
df['age_by_sex_mean'] = df.groupby('sex')['age'].transform('mean')
df['age_minus_group'] = df['age'] - df['age_by_sex_mean']
df

### 5.1 다항식 피처 자동 생성

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

X = np.array([[1, 2], [3, 4], [5, 6]])
poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)
X_poly = poly.fit_transform(X)

pd.DataFrame(X_poly, columns=poly.get_feature_names_out(['a','b']))

---

## 6. 집계(Aggregation) 피처 — 상위권 필수

트랜잭션/이력 데이터에서 **group-level 통계**를 만드는 것.

In [ ]:
# 예: 사용자별 거래 이력
tx = pd.DataFrame({
    'user_id': [1, 1, 1, 2, 2, 3, 3, 3, 3],
    'amount':  [10, 20, 15, 100, 80, 5, 8, 3, 7],
    'hour':    [9, 14, 20, 10, 11, 22, 23, 1, 2]
})

# 사용자별 집계 피처
user_agg = tx.groupby('user_id').agg(
    tx_count     = ('amount', 'count'),
    tx_sum       = ('amount', 'sum'),
    tx_mean      = ('amount', 'mean'),
    tx_std       = ('amount', 'std'),
    tx_min       = ('amount', 'min'),
    tx_max       = ('amount', 'max'),
    tx_range     = ('amount', lambda x: x.max() - x.min()),
    hour_mean    = ('hour', 'mean'),
    hour_nunique = ('hour', 'nunique')
).reset_index()
user_agg

> 💡 user_agg를 원본 테이블에 `merge`해서 사용. **이것만 잘해도 베이스라인 대비 수십 등 상승** 가능.

---

## 7. 시계열 피처

시간 데이터는 **가장 정보 많은 피처의 보고**입니다.

In [ ]:
dates = pd.to_datetime(pd.date_range('2024-01-01', periods=10, freq='D'))
df_t = pd.DataFrame({'date': dates, 'sales': np.random.randint(50, 200, 10)})

# 기본 시간 피처
df_t['year']       = df_t['date'].dt.year
df_t['month']      = df_t['date'].dt.month
df_t['day']        = df_t['date'].dt.day
df_t['dayofweek']  = df_t['date'].dt.dayofweek
df_t['is_weekend'] = (df_t['dayofweek'] >= 5).astype(int)
df_t['quarter']    = df_t['date'].dt.quarter

# 순환 피처 (sin/cos) — 월, 요일 등의 **주기성** 표현
df_t['month_sin'] = np.sin(2*np.pi * df_t['month'] / 12)
df_t['month_cos'] = np.cos(2*np.pi * df_t['month'] / 12)

df_t

In [ ]:
# Lag / Rolling 피처
df_t['sales_lag1']    = df_t['sales'].shift(1)   # 1일 전
df_t['sales_roll3']   = df_t['sales'].rolling(3).mean()
df_t['sales_diff']    = df_t['sales'].diff(1)
df_t['sales_pct_chg'] = df_t['sales'].pct_change()
df_t

> ⚠️ **시계열 CV** 를 쓸 때 future leakage 금지. `TimeSeriesSplit` 필수.

---

## 8. 텍스트 기본 피처

In [ ]:
texts = pd.DataFrame({'review': [
    "This is amazing!",
    "not bad",
    "terrible product, never buy",
    "so good so great",
]})

# 길이, 단어수
texts['len']       = texts['review'].str.len()
texts['word_cnt']  = texts['review'].str.split().str.len()
texts['upper_cnt'] = texts['review'].str.count(r'[A-Z]')
texts['excl_cnt']  = texts['review'].str.count('!')
texts

---

## 9. Data Leakage 주의사항 총정리

### 🚨 Leakage 전형 케이스
1. **미래 데이터를 학습에 사용**: 시계열에서 test 기간 데이터로 통계 계산
2. **타깃 정보가 피처에 섞임**: Target Encoding을 KFold 없이 적용
3. **Train/Test 합쳐서 StandardScaler fit**: test 통계가 학습에 반영됨
4. **target-related 피처를 만듦**: `event_happened_flag` 같은 것

### ✅ 올바른 순서 (Pipeline 사용 권장)
```
1) train/test 분리
2) train만으로 Imputer/Scaler/Encoder fit
3) 같은 transform 을 test에 적용
4) KFold 내에서 Target Encoding 계산
```

---

## 10. Feature Engineering 템플릿 (실전)

In [ ]:
def feature_engineering(df):
    df = df.copy()

    # 1. 결측치 플래그
    for col in df.columns[df.isna().any()]:
        df[f'{col}_missing'] = df[col].isna().astype(int)

    # 2. 결측치 채우기
    num_cols = df.select_dtypes(include='number').columns
    cat_cols = df.select_dtypes(include='object').columns
    for c in num_cols:
        df[c] = df[c].fillna(df[c].median())
    for c in cat_cols:
        df[c] = df[c].fillna('missing')

    # 3. 수치 변환
    for c in num_cols:
        if df[c].skew() > 1.0 and (df[c] >= 0).all():
            df[f'{c}_log1p'] = np.log1p(df[c])

    # 4. 범주 frequency encoding
    for c in cat_cols:
        df[f'{c}_freq'] = df[c].map(df[c].value_counts())

    return df

# 예시 실행
sample = pd.DataFrame({
    'age':  [25, np.nan, 35, 45, 28],
    'fare': [7.2, 100, 5, 200, 8],
    'sex':  ['M', 'F', 'M', 'F', 'M']
})
feature_engineering(sample)

---

## 📝 Day 12-14 체크리스트
- [ ] 결측치 5가지 전략 이해
- [ ] One-Hot, Label, Frequency, Target Encoding 차이 이해
- [ ] **K-Fold Target Encoding** 직접 구현 가능
- [ ] 시계열 lag/rolling 피처 생성
- [ ] Aggregation 피처로 group-level 통계 생성
- [ ] Data Leakage 발생 상황 예방

다음 주는 이 피처 위에 **Gradient Boosting (XGBoost/LightGBM/CatBoost)** 을 얹습니다.